# CHAMELEON layout student: preference training (DPO)

Starts from the v2 student (Qwen3-4B + LoRA trained on 2,659 Sonnet layouts). It teaches the student to prefer Sonnet's layout over its own on the same task:

1. The student answers about 1,000 training prompts (batched).
2. Pairs: chosen = Sonnet's layout, rejected = the student's own.
3. The SFT adapter is merged into the base on local disk, so the reference for DPO is the v2 student itself.
4. DPO with a fresh LoRA. Checkpoints go to Drive and a re-run resumes.
5. Generates the 30 held-out and 56 demo layouts.

Needs in `MyDrive/chameleon-distill/`: `train_v2.jsonl`, `eval_prompts_v2.jsonl`, `demo_prompts.jsonl`, `lora_adapter_Qwen3-4B-Instruct-2507_v2/`. **Runtime: L4.**

In [ ]:
# 1. Drive + paths
from google.colab import drive
drive.mount('/content/drive')
import os, json, glob, random
D = '/content/drive/MyDrive/chameleon-distill'
SFT = 'Qwen3-4B-Instruct-2507_v2'
ADAPTER = f'{D}/lora_adapter_{SFT}'
TAG = SFT + '_dpo'
CKPT = f'{D}/checkpoints_{TAG}'; os.makedirs(CKPT, exist_ok=True)
N_PAIRS = 1000
SYSTEM = 'You are an ad layout engine. Given a source ad as JSON and a target canvas, return the resized layout as JSON.'
for f in ['train_v2.jsonl', 'eval_prompts_v2.jsonl', 'demo_prompts.jsonl']:
    assert os.path.exists(f'{D}/{f}'), f'missing {f}'
assert os.path.isdir(ADAPTER), f'missing {ADAPTER}'

In [ ]:
# 2. Install
!pip -q install unsloth
import unsloth, trl
print('unsloth', unsloth.__version__, '| trl', trl.__version__)

In [ ]:
# 3. Load the v2 student (base + SFT adapter) for generation
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(ADAPTER, max_seq_length=4096, load_in_4bit=True)
FastLanguageModel.for_inference(model)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

def chat(prompt):
    return tokenizer.apply_chat_template([{'role':'system','content':SYSTEM},{'role':'user','content':prompt}],
                                         tokenize=False, add_generation_prompt=True, enable_thinking=False)

def generate(prompts, batch=16, max_new=1200):
    outs = []
    for i in range(0, len(prompts), batch):
        enc = tokenizer([chat(p) for p in prompts[i:i+batch]], return_tensors='pt', padding=True).to('cuda')
        gen = model.generate(**enc, max_new_tokens=max_new, do_sample=False)
        outs += tokenizer.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
        print(f'{min(i+batch, len(prompts))}/{len(prompts)}', flush=True)
    return outs

In [ ]:
# 4. The student's own answers on N_PAIRS training prompts -> preference pairs (saved to Drive; a re-run reuses them)
PAIRS = f'{D}/dpo_pairs_{SFT}.jsonl'
if not os.path.exists(PAIRS):
    rows = [json.loads(l) for l in open(f'{D}/train_v2.jsonl')]
    random.Random(0).shuffle(rows); rows = rows[:N_PAIRS]
    own = generate([r['prompt'] for r in rows])
    def norm(t):
        t = t.strip(); i = t.find('{'); return t[i:] if i >= 0 else t
    with open(PAIRS, 'w') as f:
        kept = 0
        for r, o in zip(rows, own):
            if norm(o).replace(' ', '') == r['answer'].replace(' ', ''): continue   # already identical to Sonnet
            f.write(json.dumps({'prompt': chat(r['prompt']), 'chosen': r['answer'], 'rejected': norm(o)}) + '\n'); kept += 1
    print('pairs written:', kept)
pairs = [json.loads(l) for l in open(PAIRS)]
print(len(pairs), 'pairs')

In [ ]:
# 5. Merge the SFT adapter into the base on LOCAL disk (the DPO reference = the v2 student), reload, add a fresh LoRA
import gc, torch
MERGED = '/content/merged_v2'
if not os.path.exists(f'{MERGED}/config.json'):
    model.save_pretrained_merged(MERGED, tokenizer, save_method='merged_16bit')
del model; gc.collect(); torch.cuda.empty_cache()
model, tokenizer = FastLanguageModel.from_pretrained(MERGED, max_seq_length=4096, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=0)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# 6. DPO (1 epoch, small learning rate; checkpoints on Drive)
import inspect
from datasets import Dataset
from trl import DPOTrainer, DPOConfig
try:
    from unsloth import PatchDPOTrainer; PatchDPOTrainer()
except Exception as e:
    print('PatchDPOTrainer not used:', e)
ds = Dataset.from_list(pairs).shuffle(seed=0)
cfg = dict(output_dir=CKPT, beta=0.1, learning_rate=5e-6, num_train_epochs=1,
           per_device_train_batch_size=1, gradient_accumulation_steps=8,
           lr_scheduler_type='cosine', warmup_ratio=0.1, optim='adamw_8bit', bf16=True,
           logging_steps=5, save_strategy='steps', save_steps=25, save_total_limit=3,
           max_length=3072, report_to='none', seed=0)
params = inspect.signature(DPOConfig).parameters
if 'max_prompt_length' in params: cfg['max_prompt_length'] = 2048
cfg = {k: v for k, v in cfg.items() if k in params}
args = DPOConfig(**cfg)
kw = dict(model=model, ref_model=None, args=args, train_dataset=ds)
kw['processing_class' if 'processing_class' in inspect.signature(DPOTrainer.__init__).parameters else 'tokenizer'] = tokenizer
trainer = DPOTrainer(**kw)
resume = bool(glob.glob(f'{CKPT}/checkpoint-*'))
print('resuming' if resume else 'starting fresh')
trainer.train(resume_from_checkpoint=resume)
json.dump(trainer.state.log_history, open(f'{D}/train_log_{TAG}.json', 'w'))
model.save_pretrained(f'{D}/lora_adapter_{TAG}'); tokenizer.save_pretrained(f'{D}/lora_adapter_{TAG}')
print('saved', f'lora_adapter_{TAG}')

In [ ]:
# 7. Held-out (30) and demo (56) layouts from the DPO student -> send both files back
FastLanguageModel.for_inference(model)
for name, src in [('eval_outputs', 'eval_prompts_v2.jsonl'), ('demo_outputs', 'demo_prompts.jsonl')]:
    out = f'{D}/{name}_{TAG}.jsonl'
    rows = [json.loads(l) for l in open(f'{D}/{src}')]
    done = {json.loads(l)['cell'] for l in open(out)} if os.path.exists(out) else set()
    rows = [r for r in rows if r['cell'] not in done]
    if rows:
        texts = generate([r['prompt'] for r in rows])
        with open(out, 'a') as f:
            for r, t in zip(rows, texts): f.write(json.dumps({'cell': r['cell'], 'output': t}) + '\n')
    print('done ->', out)